# Full Project Training (End-to-End)

This notebook runs the core training pipeline for the whole project using the same entrypoints that power `scripts/train_all.py`. Use the toggles below to keep a lightweight demo run or enable the full/vision-heavy steps for production training.


## Environment setup


In [7]:
from pathlib import Path
import sys

# find repo root from current notebook location
repo_root = Path.cwd()
for parent in [repo_root] + list(repo_root.parents):
    if (parent / "requirements.txt").exists() and (parent / "scripts").exists():
        repo_root = parent
        break

print("Repo root:", repo_root)
%pip install -r {repo_root / "requirements.txt"}


Repo root: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2
Ignoring tensorflow-cpu: markers 'platform_system != "Darwin" and platform_machine != "aarch64" and platform_machine != "arm64" and python_version < "3.13"' don't match your environment
Ignoring faiss-cpu: markers 'platform_system != "Darwin"' don't match your environment
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached catboost-1.2.8-cp311-cp311-macosx_11_0_universal2.whl.metadata (1.4 kB)
  Using cached shap-0.50.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (25 kB)
  Using cached lime-0.2.0.1-py3-none-any.whl
  Using cached streamlit-1.53.1-py3-none-any.whl.metadata (10 kB)
  Using cached torchaudio-2.10.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.9 kB)
  Using cached mlflow-3.8.1-py3-none-any.whl.metadata (31 kB)
  Using cached prefect-3.6.12-py3-none-any.whl.metadata (13 kB)
  Using cached transf

In [2]:
from pathlib import Path
import os
import sys
import subprocess

cwd = Path.cwd().resolve()
repo_root = cwd
if not (repo_root / "scripts" / "train_all.py").exists():
    repo_root = cwd.parents[1]
if not (repo_root / "scripts" / "train_all.py").exists():
    raise FileNotFoundError("Run this notebook from within the repo.")
print("Repo root:", repo_root)


Repo root: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2


## Dataset readiness check

These paths are required for the optional vision/voice/recommender steps. Missing paths will simply skip those steps if you leave the flags disabled.


In [3]:
required_paths = [
    repo_root / "data" / "raw" / "vision" / "Dataset",
    repo_root / "data" / "raw" / "vision" / "video",
    repo_root / "data" / "Celeb_V2",
    repo_root / "data" / "raw" / "vision" / "face_emotion",
    repo_root / "data" / "raw" / "voice",
    repo_root / "data" / "raw" / "recommendation" / "movielens.csv",
    repo_root / "data" / "raw" / "recommendation" / "items.csv",
]
for path in required_paths:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {path}")


OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/vision/Dataset
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/vision/video
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/Celeb_V2
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/vision/face_emotion
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/voice
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/recommendation/movielens.csv
OK      /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/recommendation/items.csv


## Optional: populate datasets from archives

Set `RUN_POPULATE = True` if you want the notebook to build the dataset layout from your archive folders.


In [4]:
RUN_POPULATE = False

if RUN_POPULATE:
    subprocess.run(
        [sys.executable, "scripts/populate_project_datasets.py"],
        cwd=repo_root,
        check=True,
    )


## Train everything (toggle heavy steps as needed)

`train_all.py` already runs fraud/cyber/behavior/fusion + voice + recommender. Add flags here for vision, brand, temporal video, and face-emotion models.


In [5]:
WITH_VISION = False
WITH_VISION_FULL = False
WITH_BRAND = False
WITH_VIDEO_TEMPORAL = False
WITH_FACE_EMOTION = False
VOICE_LIMIT_PER_CLASS = 200  # 0 uses all samples

cmd = [
    sys.executable,
    "scripts/train_all.py",
    "--voice-limit-per-class",
    str(VOICE_LIMIT_PER_CLASS),
]
if WITH_VISION:
    cmd.append("--with-vision")
if WITH_VISION_FULL:
    cmd.append("--with-vision-full")
if WITH_BRAND:
    cmd.append("--with-brand")
if WITH_VIDEO_TEMPORAL:
    cmd.append("--with-video-temporal")
if WITH_FACE_EMOTION:
    cmd.append("--with-face-emotion")

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=repo_root, check=True)


Running: /opt/anaconda3/envs/automl311/bin/python scripts/train_all.py --voice-limit-per-class 200

$ /opt/anaconda3/envs/automl311/bin/python src/scripts/run_fraud_experiment.py
Project root: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2
Loaded raw fraud data with shape: (284807, 31)
Feature table shape: (284807, 35)
Train shape: (170884, 34), Val shape: (56961, 34), Test shape: (56962, 34)
Validation metrics (supervised model):
  roc_auc: 0.8916
  pr_auc: 0.5226
  f1: 0.6636
  precision: 0.5984
  recall: 0.7449
  accuracy: 0.9987
Test metrics (supervised model):
  roc_auc: 0.8920
  pr_auc: 0.5918
  f1: 0.7080
  precision: 0.6299
  recall: 0.8081
  accuracy: 0.9988
Test metrics (hybrid supervised + anomaly):
  roc_auc: 0.9581
  pr_auc: 0.6225
  f1: 0.7080
  precision: 0.6299
  recall: 0.8081
  accuracy: 0.9988
Experiment completed. Plots saved to: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/experiments/fraud/plots
Artifacts saved under: 

/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/src/uais/data/load_cyber_data.py:120: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, **read_kwargs)


[warn] Skipping UNSW-NB15_1.csv (no label columns)
Loading UNSW-NB15_2.csv ...


/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/src/uais/data/load_cyber_data.py:120: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, **read_kwargs)


[warn] Skipping UNSW-NB15_2.csv (no label columns)
Loading UNSW-NB15_3.csv ...
[warn] Skipping UNSW-NB15_3.csv (no label columns)
Loading UNSW-NB15_4.csv ...
[warn] Skipping UNSW-NB15_4.csv (no label columns)
Loading UNSW-NB15_LIST_EVENTS.csv ...
[warn] Skipping UNSW-NB15_LIST_EVENTS.csv (no label columns)
Loading UNSW_NB15_testing-set.csv ...
Loading UNSW_NB15_training-set.csv ...
Full cyber raw shape: (257673, 45)
Numeric features: 44, Categorical features: 3
Cyber feature table shape: (257673, 48)


...

/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/src/uais/data/load_cyber_data.py:120: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, **read_kwargs)


[warn] Skipping UNSW-NB15_2.csv (no label columns)
Loading UNSW-NB15_3.csv ...
[warn] Skipping UNSW-NB15_3.csv (no label columns)
Loading UNSW-NB15_4.csv ...
[warn] Skipping UNSW-NB15_4.csv (no label columns)
Loading UNSW-NB15_LIST_EVENTS.csv ...
[warn] Skipping UNSW-NB15_LIST_EVENTS.csv (no label columns)
Loading UNSW_NB15_testing-set.csv ...
Loading UNSW_NB15_training-set.csv ...
Full cyber raw shape: (257673, 45)
Numeric features: 44, Categorical features: 3
Cyber feature table shape: (257673, 48)
Loaded LDAP behavior directory: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/data/raw/behavior/r4.2/LDAP -> shape (16743, 10)
Numeric features: 1, Categorical features: 8
Behavior feature table shape: (16743, 10)


2026-01-22 17:23:48,818 [INFO] uais.anomaly.train_autoencoder: Saved autoencoder for behavior to /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/models/behavior/behavior_autoencoder.pkl
2026-01-22 17:23:48,862 [INFO] uais.fusion.train_fusion_model: Saved fusion meta-model to /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/models/fusion/fusion_meta_model.pkl
2026-01-22 17:23:48,863 [INFO] __main__: Saved /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/experiments/fusion/metrics/metrics.json
2026-01-22 17:23:48,875 [INFO] __main__: Saved fusion scores to /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/experiments/fusion/fusion_scores.csv
2026-01-22 17:23:48,875 [INFO] __main__: Fusion experiment complete



$ /opt/anaconda3/envs/automl311/bin/python app/models/voice/emotion_train.py --limit-per-class 200
📊 Using 1000 valid samples (filtered 0 bad samples)
📊 Features per sample: 31 (26 MFCC + 5 audio features)
✅ Trained voice emotion model (accuracy=42.50%)
📁 Saved to models/voice_emotion.pkl

$ /opt/anaconda3/envs/automl311/bin/python src/train/train_recommender.py
Saved recommender model to models/recommender/recommender_model.pkl
Saved recommender meta to models/recommender/recommender_meta.joblib
Metrics written to experiments/recommender/metrics/metrics.json

✅ Training complete. Check experiments/*/metrics and models/* outputs.


CompletedProcess(args=['/opt/anaconda3/envs/automl311/bin/python', 'scripts/train_all.py', '--voice-limit-per-class', '200'], returncode=0)

## Artifacts summary

Quick peek at the output directories after training finishes.


In [6]:
for folder in ["models", "experiments", "artifacts", "runs"]:
    path = repo_root / folder
    if not path.exists():
        continue
    print(f"\n{folder}/")
    for item in sorted(path.iterdir())[:20]:
        print(" -", item.name)



models/
 - behavior
 - behavior.pkl
 - brand
 - cyber
 - fraud
 - fusion
 - nlp
 - recommender
 - vision
 - voice_emotion.pkl
 - voice_emotion_nn.pt

experiments/
 - behavior
 - cyber
 - fraud
 - fusion
 - generative
 - recommender
 - report_summary.json
 - vision

artifacts/
 - README.md
 - brand
 - vision_temporal

runs/
 - detect
 - mlflow
